In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"
# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)



In [2]:
# load the actual model to make sure it works
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed via HuggingFace transformers (ground truth)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
s   = "Explain MoE in transformers in 3 sentences."
ids = processor.tokenizer(s, return_tensors="pt")["input_ids"]

embed_tokens = model.get_input_embeddings()          # the nn.Embedding itself
theirs       = embed_tokens(ids)

print(f"weight : {tuple(embed_tokens.weight.shape)}  dtype={embed_tokens.weight.dtype}")
print(f"ids    : {tuple(ids.shape)}  {ids[0, :10].tolist()}")
print(f"vecs   : {tuple(theirs.shape)}")
print(f"theirs[0, 0, :6] = {theirs[0, 0, :6].tolist()}")

weight : (262144, 1536)  dtype=torch.bfloat16
ids    : (1, 10)  [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761]
vecs   : (1, 10, 1536)
theirs[0, 0, :6] = [-0.11279296875, 0.18359375, -0.08251953125, -0.7265625, 2.171875, -1.484375]


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Multimodal embedding — local test_data (Moon.jpg + Chats.mp3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "path": "test_data/Moon.jpg"},
        {"type": "audio", "path": "test_data/Chats.mp3"},
        {"type": "text",  "text": "Describe the image and the audio."},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

fused = out.hidden_states[0]   # fused residual stream: text + image + audio tokens

print("inputs keys :", list(inputs.keys()))
for k, v in inputs.items():
    if torch.is_tensor(v):
        print(f"  {k:20s} {tuple(v.shape)}")
print(f"fused       : {tuple(fused.shape)}  dtype={fused.dtype}")
print(f"fused[0,  0, :6] = {fused[0,  0, :6].tolist()}   # first token")
print(f"fused[0, -1, :6] = {fused[0, -1, :6].tolist()}   # last token")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/gemma4/feature_extraction_gemma4.py:208: RuntimeWarning: divide by zero encountered in matmul
  mel_spec = np.matmul(magnitude_spec, self.mel_filters)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/g

inputs keys : ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'input_features', 'input_features_mask']
  input_ids            (1, 415)
  attention_mask       (1, 415)
  mm_token_type_ids    (1, 415)
  pixel_values         (1, 2520, 768)
  image_position_ids   (1, 2520, 2)
  input_features       (1, 537, 128)
  input_features_mask  (1, 537)
fused       : (1, 415, 1536)  dtype=torch.bfloat16
fused[0,  0, :6] = [-1.640625, -1.53125, 0.1884765625, -1.484375, -0.99609375, -0.0272216796875]   # first token
fused[0, -1, :6] = [1.6015625, -1.828125, -0.36328125, 0.83203125, 0.0030364990234375, 0.014404296875]   # last token


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Extract image/audio soft-token slices from the fused residual stream
#  so we can compare against our projector outputs in load_pytorch_model.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tt = inputs["mm_token_type_ids"][0]      # (seq,)
uniq, counts = torch.unique(tt, return_counts=True)
print("mm_token_type_ids unique:", dict(zip(uniq.tolist(), counts.tolist())))

for t in uniq.tolist():
    if t == 0:          # text
        continue
    idx   = (tt == t).nonzero(as_tuple=True)[0]
    slice_ = fused[0, idx]     # (n_tokens, 1536)
    label  = {1: "image", 2: "audio", 3: "audio"}.get(t, f"type={t}")
    print(f"\n{label}  type_id={t}  → {tuple(slice_.shape)}  dtype={slice_.dtype}")
    print(f"  first token : {slice_[0, :6].tolist()}")


mm_token_type_ids unique: {0: 20, 1: 260, 3: 135}

image  type_id=1  → (260, 1536)  dtype=torch.bfloat16
  first token : [0.9453125, 0.1572265625, -0.88671875, -1.5546875, -0.859375, 0.314453125]

audio  type_id=3  → (135, 1536)  dtype=torch.bfloat16
  first token : [-1.90625, 0.037841796875, -3.8125, -1.3203125, -1.7578125, 0.8046875]
